# Feature Selection – Research & Production Ready

This notebook implements **filter, embedded, wrapper, and ensemble feature selection**.
Designed for **binary or multiclass classification**, leakage-free, CV-safe.

In [9]:

# ============================
# 1. Imports
# ============================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import shap
import matplotlib.pyplot as plt
from pathlib import Path

In [10]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

## 2. Load Your Dataset

In [11]:
TRAIN_DATA_PATH = Path("../data/processed/nsl_kdd_selected_features_train.csv")
TEST_DATA_PATH = Path("../data/processed/nsl_kdd_selected_features_test.csv")
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)

In [12]:
train_df.head()

,duration,src_bytes,dst_bytes,total_bytes,byte_ratio,inbound_outbound_ratio,count,srv_count,diff_srv_rate,same_srv_rate,connection_intensity_ratio,dst_host_count,dst_host_srv_count,dst_host_diff_srv_rate,dst_host_same_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,serror_rate,srv_serror_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,combined_error_rate,logged_in,hot,num_failed_logins,root_shell,is_guest_login,protocol_type,service,flag,login_anomaly_score,privileged_ratio,service_rarity,command_density,is_attack_binary,is_attack_multiclass
0,0.0,491,0,491,491.000000,0.000000,2,2,0.00,1.00,0.666667,150,25,0.03,0.17,0.17,0.00,0.0,0.0,0.00,0.00,0.05,0.0,0,0,0,0,0,tcp,ftp_data,SF,0.0,0.0,18.363411,0.0,0,normal
1,0.0,146,0,146,146.000000,0.000000,13,1,0.15,0.08,6.500000,255,1,0.60,0.00,0.88,0.00,0.0,0.0,0.00,0.00,0.00,0.0,0,0,0,0,0,udp,other,SF,0.0,0.0,28.899518,0.0,0,normal
2,0.0,0,0,0,0.000000,0.000000,123,6,0.07,0.05,17.571429,255,26,0.05,0.10,0.00,0.00,1.0,1.0,1.00,1.00,0.00,0.5,0,0,0,0,0,tcp,private,S0,0.0,0.0,5.764563,0.0,1,dos
3,0.0,232,8153,8385,0.028452,34.991416,5,5,0.00,1.00,0.833333,30,255,0.00,1.00,0.03,0.04,0.2,0.2,0.03,0.01,0.00,0.1,1,0,0,0,0,tcp,http,SF,0.0,0.0,3.122936,0.0,0,normal
4,0.0,199,420,619,0.472684,2.100000,30,32,0.00,1.00,0.909091,255,255,0.00,1.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.0,1,0,0,0,0,tcp,http,SF,0.0,0.0,3.122936,0.0,0,normal


In [13]:
train_df.columns

Index(['duration', 'src_bytes', 'dst_bytes', 'total_bytes', 'byte_ratio',
       'inbound_outbound_ratio', 'count', 'srv_count', 'diff_srv_rate',
       'same_srv_rate', 'connection_intensity_ratio', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_diff_srv_rate',
       'dst_host_same_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'serror_rate', 'srv_serror_rate',
       'dst_host_serror_rate', 'dst_host_srv_serror_rate',
       'dst_host_rerror_rate', 'combined_error_rate', 'logged_in', 'hot',
       'num_failed_logins', 'root_shell', 'is_guest_login', 'protocol_type',
       'service', 'flag', 'login_anomaly_score', 'privileged_ratio',
       'service_rarity', 'command_density', 'is_attack_binary',
       'is_attack_multiclass'],
      dtype='object')

In [ ]:
# We want to drop the multiclass column from features
X = train_df.drop(columns=['is_attack_multiclass', 'is_attack_binary','label','difficulty_level'])  # only features remain
y = train_df['is_attack_binary']  # target is binary

# Split into train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,  # preserves class ratio
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (100778, 54) (100778,)
Test shape: (25195, 54) (25195,)


In [ ]:
categorical_cols = ['protocol_type', 'service', 'flag']
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(train_df['is_attack_multiclass'])  # convert multiclass to 0,1,2...

In [24]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: Index(['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
       'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
       'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
       'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'connection_intensity_ratio', 'total_bytes',
       'byte_ratio', 'combined_error_rate', 'service_diversity_score',
       'host_diversity', 'traffic_burst', 'privilege_attempt',
       'login_anomaly_score', 'privi

In [25]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_cat = pd.DataFrame(
    ohe.fit_transform(X_train[categorical_cols]),
    columns=ohe.get_feature_names_out(categorical_cols),
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    ohe.transform(X_test[categorical_cols]),
    columns=ohe.get_feature_names_out(categorical_cols),
    index=X_test.index
)


In [26]:
X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

print(X_train_final.shape)

(100778, 134)


## 3. Filter Methods

In [28]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif

# Variance Threshold
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X_train_final)
vt_features = X_train_final.columns[vt.get_support()]

print("Variance Threshold selected features:", vt_features)

# Mutual Information
mi = mutual_info_classif(X_train_final, y_train, random_state=42)
mi_scores = pd.Series(mi, index=X_train_final.columns).sort_values(ascending=False)
print("Top 20 features by mutual info:\n", mi_scores.head(20))

Variance Threshold selected features: Index(['duration', 'src_bytes', 'dst_bytes', 'wrong_fragment', 'hot',
       'logged_in', 'num_compromised', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'connection_intensity_ratio', 'total_bytes',
       'byte_ratio', 'combined_error_rate', 'service_diversity_score',
       'login_anomaly_score', 'service_rarity', 'inbound_outbound_ratio',
       'protocol_type_icmp', 'protocol_type_tcp', 'protocol_type_udp',
       'service_domain_u', 'service_eco_i', 'service_ecr_i', 'service_finger',
       'service_ftp', 'service_ftp_data', 'service_http', 's

## 4. Embedded Methods

In [29]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)

l1_pipe = LogisticRegression(
    penalty="l1", solver="saga", C=0.1, max_iter=5000, random_state=42
)
l1_pipe.fit(X_train_scaled, y_train)

l1_scores = pd.Series(
    np.abs(l1_pipe.coef_).mean(axis=0),
    index=X_train_final.columns
).sort_values(ascending=False)

print("Top 20 features (L1 Logistic Regression):")
print(l1_scores.head(20))

Top 20 features (L1 Logistic Regression):
service_http                   1.663831
service_private                1.577919
count                          1.523873
srv_serror_rate                1.475645
dst_host_same_src_port_rate    1.324371
dst_host_srv_serror_rate       1.153371
flag_S0                        1.141514
dst_host_srv_count             1.139236
dst_host_rerror_rate           1.011019
flag_S1                        0.878785
wrong_fragment                 0.825447
same_srv_rate                  0.821575
hot                            0.754718
byte_ratio                     0.733617
service_smtp                   0.732476
login_anomaly_score            0.661407
flag_SF                        0.655290
flag_RSTR                      0.634708
src_bytes                      0.621080
service_domain_u               0.587516
dtype: float64


In [30]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500, random_state=42, n_jobs=-1, class_weight="balanced"
)
rf.fit(X_train_final, y_train)

rf_scores = pd.Series(rf.feature_importances_, index=X_train_final.columns).sort_values(ascending=False)

print("Top 20 features (Random Forest):")
print(rf_scores.head(20))

Top 20 features (Random Forest):
total_bytes                    0.102280
src_bytes                      0.082691
inbound_outbound_ratio         0.082023
dst_bytes                      0.065071
connection_intensity_ratio     0.050803
combined_error_rate            0.048334
byte_ratio                     0.047499
service_diversity_score        0.044131
flag_SF                        0.036459
dst_host_srv_count             0.035970
dst_host_same_srv_rate         0.029523
same_srv_rate                  0.028539
diff_srv_rate                  0.025950
count                          0.023793
logged_in                      0.021440
service_rarity                 0.021017
dst_host_same_src_port_rate    0.020130
dst_host_diff_srv_rate         0.018599
srv_serror_rate                0.016906
protocol_type_icmp             0.015286
dtype: float64


## 5. Permutation Importance

In [31]:
from sklearn.inspection import permutation_importance
import pandas as pd

perm = permutation_importance(
    rf,
    X_train_final,
    y_train,
    scoring="f1_macro",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_scores = pd.Series(
    perm.importances_mean,
    index=X_train_final.columns
).sort_values(ascending=False)

print("Top 20 Permutation Importance Features:")
print(perm_scores.head(20))

Top 20 Permutation Importance Features:
dst_host_srv_count             0.000211
duration                       0.000156
dst_host_same_src_port_rate    0.000111
dst_host_count                 0.000096
byte_ratio                     0.000076
command_density                0.000058
dst_host_srv_diff_host_rate    0.000057
src_bytes                      0.000052
inbound_outbound_ratio         0.000048
total_bytes                    0.000047
service_rarity                 0.000046
dst_host_diff_srv_rate         0.000046
dst_host_rerror_rate           0.000045
service_telnet                 0.000039
service_diversity_score        0.000038
dst_host_same_srv_rate         0.000037
service_private                0.000026
dst_host_srv_serror_rate       0.000023
logged_in                      0.000019
connection_intensity_ratio     0.000019
dtype: float64


## 6. SHAP Importance

In [32]:
# =========================
# FAST SHAP EXPLANATION (OPTIMIZED)
# =========================

import shap
import numpy as np
import pandas as pd

# ------------------------- 
# 1. SUBSAMPLE DATA (CRITICAL)
# -------------------------
SHAP_SAMPLE_SIZE = 2000  # 500–2000 is ideal

if len(X_train_final) > SHAP_SAMPLE_SIZE:
    X_shap = X_train_final.sample(
        n=SHAP_SAMPLE_SIZE,
        random_state=42
    )
else:
    X_shap = X_train_final.copy()

print("SHAP sample shape:", X_shap.shape)

# -------------------------
# 2. USE TREE EXPLAINER (FAST MODE)
# -------------------------
explainer = shap.TreeExplainer(
    rf,
    feature_perturbation="tree_path_dependent"  # fastest for RF
)

# -------------------------
# 3. COMPUTE SHAP VALUES
# -------------------------
shap_values = explainer.shap_values(X_shap)

# -------------------------
# 4. HANDLE BINARY vs MULTICLASS
# -------------------------
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # New SHAP: (n_samples, n_features, n_classes)
    shap_scores = np.mean(
        np.abs(shap_values[:, :, 1]),  # ATTACK class
        axis=0
    )

elif isinstance(shap_values, list):
    # Old SHAP: [class0, class1]
    shap_scores = np.mean(
        np.abs(shap_values[1]),
        axis=0
    )

else:
    raise ValueError("Unexpected SHAP output format")

# -------------------------
# 5. FEATURE IMPORTANCE
# -------------------------
shap_scores = pd.Series(
    shap_scores,
    index=X_shap.columns
).sort_values(ascending=False)

print("\nTop 20 SHAP Important Features:")
print(shap_scores.head(20))

SHAP sample shape: (2000, 134)

Top 20 SHAP Important Features:
total_bytes                   0.061644
src_bytes                     0.051682
inbound_outbound_ratio        0.042996
dst_bytes                     0.037676
combined_error_rate           0.026883
service_diversity_score       0.024736
connection_intensity_ratio    0.024506
dst_host_srv_count            0.021610
byte_ratio                    0.019422
dst_host_same_srv_rate        0.017746
logged_in                     0.017684
flag_SF                       0.015621
count                         0.014922
same_srv_rate                 0.014271
diff_srv_rate                 0.013091
service_rarity                0.011713
dst_host_diff_srv_rate        0.011212
dst_host_serror_rate          0.010822
srv_serror_rate               0.010481
service_private               0.010206
dtype: float64


## 8. Ensemble Feature Selection

In [37]:
# Ensure all scores use same index
common_features = (
    mi_scores.index
    .intersection(l1_scores.index)
    .intersection(rf_scores.index)
    .intersection(perm_scores.index)
    .intersection(shap_scores.index)
)

mi_scores   = mi_scores.loc[common_features]
l1_scores   = l1_scores.loc[common_features]
rf_scores   = rf_scores.loc[common_features]
perm_scores = perm_scores.loc[common_features]
shap_scores = shap_scores.loc[common_features]

In [38]:
fi_df = pd.DataFrame({
    "mi": mi_scores,
    "l1": l1_scores,
    "rf": rf_scores,
    "perm": perm_scores,
    "shap": shap_scores
})

In [43]:
fi_df = fi_df.div(fi_df.sum(axis=0), axis=1)
fi_df["final_score"] = fi_df.mean(axis=1)
fi_df = fi_df.sort_values("final_score", ascending=False)
fi_df

,mi,l1,rf,perm,shap,final_score
total_bytes,0.058526,0.006066,1.022798e-01,3.435699e-02,1.055835e-01,6.136245e-02
dst_host_srv_count,0.034504,0.033185,3.596992e-02,1.549719e-01,3.701347e-02,5.912879e-02
src_bytes,0.058130,0.018092,8.269106e-02,3.801197e-02,8.852047e-02,5.708896e-02
inbound_outbound_ratio,0.044813,0.002916,8.202324e-02,3.508800e-02,7.364331e-02,4.769677e-02
byte_ratio,0.058762,0.021370,4.749939e-02,5.555599e-02,3.326602e-02,4.329071e-02
dst_bytes,0.045219,0.009617,6.507062e-02,2.924106e-03,6.453136e-02,3.747245e-02
dst_host_same_src_port_rate,0.013609,0.038578,2.012989e-02,8.114037e-02,1.709539e-02,3.411053e-02
service_diversity_score,0.038501,0.013859,4.413100e-02,2.777753e-02,4.236777e-02,3.332713e-02
connection_intensity_ratio,0.046513,0.000000,5.080277e-02,1.388851e-02,4.197320e-02,3.063554e-02
combined_error_rate,0.036200,0.015153,4.833431e-02,-1.461842e-03,4.604429e-02,2.885399e-02


## 9. Select Best Features (95% cumulative importance)

In [40]:

cum_imp = fi_df["final_score"].cumsum()
selected_features = cum_imp[cum_imp <= 0.95].index.tolist()

print("Selected Features:", selected_features)


Selected Features: ['total_bytes', 'dst_host_srv_count', 'src_bytes', 'inbound_outbound_ratio', 'byte_ratio', 'dst_bytes', 'dst_host_same_src_port_rate', 'service_diversity_score', 'connection_intensity_ratio', 'combined_error_rate', 'count', 'duration', 'service_rarity', 'dst_host_same_srv_rate', 'flag_SF', 'same_srv_rate', 'dst_host_count', 'logged_in', 'dst_host_srv_serror_rate', 'service_private', 'srv_serror_rate', 'dst_host_diff_srv_rate', 'service_http', 'dst_host_srv_diff_host_rate', 'diff_srv_rate', 'flag_S0', 'dst_host_rerror_rate', 'dst_host_serror_rate', 'command_density', 'serror_rate', 'protocol_type_icmp', 'service_telnet', 'srv_count', 'login_anomaly_score', 'srv_diff_host_rate', 'hot', 'service_ecr_i', 'wrong_fragment', 'service_domain_u', 'flag_S1', 'service_smtp', 'protocol_type_udp', 'is_guest_login', 'flag_RSTR', 'flag_REJ', 'protocol_type_tcp', 'service_ftp_data', 'service_eco_i', 'dst_host_srv_rerror_rate', 'service_other', 'service_imap4', 'rerror_rate', 'num_co

In [ ]:
# Updated selected features including engineered security features
selected_features = [
    
]

# Target columns
target_cols = ['is_attack_binary', 'is_attack_multiclass']

# Final train/test DataFrames
train_final_df = train_df[selected_features + target_cols]
test_final_df = test_df[selected_features + target_cols]

In [51]:
# =========================
# Final Feature Selection (90% cumulative importance)
# =========================

# Safety: clip negative permutation importance (best practice)
fi_df["perm"] = fi_df["perm"].clip(lower=0)

# Normalize again (optional but safe if perm was clipped)
fi_df = fi_df.div(fi_df.sum(axis=0), axis=1)

# Final ensemble score
fi_df["final_score"] = fi_df.mean(axis=1)

# Sort by importance
fi_df_sorted = fi_df.sort_values("final_score", ascending=False)

# Cumulative importance
fi_df_sorted["cumulative_importance"] = fi_df_sorted["final_score"].cumsum()

# Select features explaining 90% importance
selected_features = fi_df_sorted[
    fi_df_sorted["cumulative_importance"] <= 0.95
].index.tolist()

print(f"Selected Features Count: {len(selected_features)}")

selected_features


Selected Features Count: 54


['total_bytes',
 'dst_host_srv_count',
 'src_bytes',
 'inbound_outbound_ratio',
 'byte_ratio',
 'dst_bytes',
 'dst_host_same_src_port_rate',
 'service_diversity_score',
 'connection_intensity_ratio',
 'combined_error_rate',
 'count',
 'service_rarity',
 'duration',
 'dst_host_same_srv_rate',
 'flag_SF',
 'same_srv_rate',
 'dst_host_count',
 'logged_in',
 'srv_serror_rate',
 'dst_host_srv_serror_rate',
 'service_private',
 'dst_host_diff_srv_rate',
 'service_http',
 'diff_srv_rate',
 'dst_host_srv_diff_host_rate',
 'flag_S0',
 'dst_host_rerror_rate',
 'dst_host_serror_rate',
 'command_density',
 'serror_rate',
 'protocol_type_icmp',
 'service_telnet',
 'srv_count',
 'login_anomaly_score',
 'srv_diff_host_rate',
 'hot',
 'service_ecr_i',
 'wrong_fragment',
 'service_domain_u',
 'flag_S1',
 'service_smtp',
 'protocol_type_udp',
 'is_guest_login',
 'flag_RSTR',
 'protocol_type_tcp',
 'flag_REJ',
 'service_ftp_data',
 'service_eco_i',
 'dst_host_srv_rerror_rate',
 'service_other',
 'service

In [14]:
# =========================
# ORIGINAL FEATURE MAPPING
# =========================

# -------------------------
# Numeric (final approved)
# -------------------------
original_numeric_features = [
    'duration',
    'src_bytes',
    'dst_bytes',
    'count',
    'srv_count',
    'same_srv_rate',
    'diff_srv_rate',
    'dst_host_count',
    'dst_host_srv_count',
    'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'serror_rate',
    'srv_serror_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate',
    'dst_host_rerror_rate',
    'logged_in',
    'hot',
    'is_guest_login'
]

# -------------------------
# Categorical (original)
# -------------------------
original_categorical_features = [
    'protocol_type',
    'service',
    'flag'
]

# ---------
# 3) Engineered security features (derived)
# ---------
engineered_features = [
    'total_bytes',                  # src_bytes + dst_bytes
    'byte_ratio',                   # src_bytes / (dst_bytes + 1)
    'inbound_outbound_ratio',        # dst_bytes / (src_bytes + 1)
    'connection_intensity_ratio',    # count / (srv_count + 1)
    'combined_error_rate',           # serror_rate + rerror_rate
    'service_rarity',                # inverse service frequency
    'command_density',               # hot / (duration + 1)
    'login_anomaly_score'            # function of login flags
]

# =========================
# FINAL ORIGINAL FEATURE SET
# =========================
selected_features_original = (
    original_numeric_features +
    original_categorical_features +
    engineered_features
)

# =========================
# TARGET COLUMNS
# =========================
target_cols = ['is_attack_binary', 'is_attack_multiclass']


# =========================
# FINAL TRAIN / TEST DATAFRAMES
# =========================
train_final_df = train_df[selected_features_original + target_cols]
test_final_df = test_df[selected_features_original + target_cols]


# =========================
# SUMMARY
# =========================
print("===== FINAL FEATURE SUMMARY =====")
print("Original Numeric      :", len(original_numeric_features))
print("Original Categorical  :", len(original_categorical_features))
print("Engineered Features   :", len(engineered_features))
print("TOTAL FEATURES        :", len(selected_features_original))
print("Train shape           :", train_final_df.shape)
print("Test shape            :", test_final_df.shape)


===== FINAL FEATURE SUMMARY =====
Original Numeric      : 21
Original Categorical  : 3
Engineered Features   : 8
TOTAL FEATURES        : 32
Train shape           : (125973, 34)
Test shape            : (22544, 34)


## 10. Save Selected Features

In [15]:
train_final_df.to_csv("../data/processed/nsl_kdd_selected_features_train.csv", index=False)

In [16]:
train_final_df.to_csv("../data/processed/nsl_kdd_selected_features_train.csv", index=False)